In [ ]:
!pip install -U "bitsandbytes>=0.46.1"
!pip uninstall -y bitsandbytes
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "microsoft/Phi-4-mini-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading Phi-4-mini in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

print("\nPhi-4-mini loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

Loading tokenizer...


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading Phi-4-mini in 4-bit...


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]


✅ Phi-4-mini loaded successfully!
GPU: Tesla T4


In [ ]:
# ============================================================
# LOAD AND VALIDATE THE 30 REPLICATION INPUTS
# ============================================================

import os
import pandas as pd
import numpy as np
import torch
import gc
from pathlib import Path

INPUT_FILE = "/content/Replication_30_Inputs.xlsx"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input Excel not found: {INPUT_FILE}\n"
        "Upload the 30-input Excel file to Colab first."
    )

df_inputs = pd.read_excel(INPUT_FILE)

print("=" * 80)
print("30-INPUT REPLICATION DATA")
print("=" * 80)

print("Rows:", len(df_inputs))
print("Columns:")
print(df_inputs.columns.tolist())

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "case_id",
    "condition"
]

missing = [c for c in REQUIRED_COLUMNS if c not in df_inputs.columns]

if missing:
    raise ValueError(
        f" Missing required columns: {missing}\n"
        f"Available columns: {df_inputs.columns.tolist()}"
    )

# ------------------------------------------------------------
# Find response/input text column
# ------------------------------------------------------------

possible_text_columns = [
    "input_text",
    "prompt",
    "case_text",
    "clinical_documentation",
    "documentation",
    "medical_documentation",
    "input"
]

TEXT_COLUMN = None

for col in possible_text_columns:
    if col in df_inputs.columns:
        TEXT_COLUMN = col
        break

if TEXT_COLUMN is None:
    raise ValueError(
        " Could not identify the clinical input-text column.\n"
        "Please rename the column containing the case/documentation to "
        "'input_text'."
    )

print(f"\nClinical input column: {TEXT_COLUMN}")

# ------------------------------------------------------------
# Basic cleaning
# ------------------------------------------------------------

df_inputs["condition"] = (
    df_inputs["condition"]
    .astype(str)
    .str.strip()
)

df_inputs[TEXT_COLUMN] = (
    df_inputs[TEXT_COLUMN]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# Validate conditions
# ------------------------------------------------------------

EXPECTED_CONDITIONS = ["Neutral", "Implicit", "Explicit"]

condition_counts = df_inputs["condition"].value_counts()

print("\nConditions:")
print(condition_counts)

for condition in EXPECTED_CONDITIONS:
    count = (df_inputs["condition"] == condition).sum()

    if count != 10:
        raise ValueError(
            f" {condition}: expected 10 inputs, found {count}"
        )

# ------------------------------------------------------------
# Validate total
# ------------------------------------------------------------

if len(df_inputs) != 30:
    raise ValueError(
        f" Expected exactly 30 inputs, found {len(df_inputs)}"
    )

# ------------------------------------------------------------
# Validate 10 unique cases
# ------------------------------------------------------------

n_cases = df_inputs["case_id"].nunique()

if n_cases != 10:
    raise ValueError(
        f" Expected 10 unique cases, found {n_cases}"
    )

# ------------------------------------------------------------
# Validate each case has 3 conditions
# ------------------------------------------------------------

case_condition_counts = (
    df_inputs.groupby("case_id")["condition"]
    .nunique()
)

bad_cases = case_condition_counts[
    case_condition_counts != 3
]

if len(bad_cases) > 0:
    raise ValueError(
        " Some cases do not contain all 3 conditions:\n"
        f"{bad_cases}"
    )

# ------------------------------------------------------------
# Remove completely empty clinical inputs
# ------------------------------------------------------------

empty_inputs = (
    df_inputs[TEXT_COLUMN]
    .str.strip()
    .eq("")
)

if empty_inputs.any():
    raise ValueError(
        f" {empty_inputs.sum()} inputs have empty clinical documentation."
    )

# ------------------------------------------------------------
# Reset index
# ------------------------------------------------------------

df_inputs = df_inputs.reset_index(drop=True)

print("\n" + "=" * 80)
print("30 INPUTS VALIDATED SUCCESSFULLY")
print("=" * 80)

print("Cases      :", df_inputs["case_id"].nunique())
print("Inputs     :", len(df_inputs))

print("\nConditions:")
print(df_inputs["condition"].value_counts())

display(
    df_inputs[
        ["case_id", "condition"]
        + (
            ["case_label"]
            if "case_label" in df_inputs.columns
            else []
        )
    ].head(30)
)

30-INPUT REPLICATION DATA
Rows: 30
Columns:
['input_id', 'case_id', 'case_label', 'condition', 'original_diagnosis', 'case_text', 'full_model_input']

Clinical input column: case_text

Conditions:
condition
Neutral     10
Implicit    10
Explicit    10
Name: count, dtype: int64

✅30 INPUTS VALIDATED SUCCESSFULLY
Cases      : 10
Inputs     : 30

Conditions:
condition
Neutral     10
Implicit    10
Explicit    10
Name: count, dtype: int64


,case_id,condition,case_label
0,CASE_01,Neutral,Schizophrenia/psychosisA
1,CASE_01,Implicit,Schizophrenia/psychosisA
2,CASE_01,Explicit,Schizophrenia/psychosisA
3,CASE_02,Neutral,DEPRESSIONA
4,CASE_02,Implicit,DEPRESSIONA
5,CASE_02,Explicit,DEPRESSIONA
6,CASE_03,Neutral,ANXIETYA
7,CASE_03,Implicit,ANXIETYA
8,CASE_03,Explicit,ANXIETYA
9,CASE_04,Neutral,Schizophrenia/psychosisB


In [ ]:
# ============================================================
# PHI-4-MINI — CHECKPOINTED 30-RESPONSE GENERATION
# FIXED NaN / EMPTY-CELL BUG
# ============================================================

import os
import gc
import torch
import pandas as pd

MODEL_NAME = "Gemma-3-4B"
INPUT_FILE = "/content/Replication_30_Inputs.xlsx"
OUTPUT_FILE = "/content/Phi-4-Mini_checkpoint.xlsx"

EXPECTED_ROWS = 30

# ============================================================
# 1. LOAD THE 30 INPUTS
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f" Input file not found:\n{INPUT_FILE}"
    )

df_inputs = pd.read_excel(INPUT_FILE)

print("=" * 80)
print("INPUT VALIDATION")
print("=" * 80)

print("Rows:", len(df_inputs))
print("Columns:", list(df_inputs.columns))

if len(df_inputs) != EXPECTED_ROWS:
    raise ValueError(
        f" Expected exactly {EXPECTED_ROWS} inputs, "
        f"but found {len(df_inputs)}."
    )

required_columns = [
    "case_id",
    "condition"
]

for col in required_columns:
    if col not in df_inputs.columns:
        raise ValueError(
            f" Required column missing: {col}"
        )

print(" 30 inputs loaded successfully")

print("\nConditions:")
print(df_inputs["condition"].value_counts())

# ============================================================
# 2. FIND CASE-TEXT COLUMN
# ============================================================

possible_text_columns = [
    "case_text",
    "medical_documentation",
    "documentation",
    "patient_case",
    "input_text",
    "prompt_input",
    "case"
]

TEXT_COLUMN = None

for col in possible_text_columns:
    if col in df_inputs.columns:
        TEXT_COLUMN = col
        break

if TEXT_COLUMN is None:
    raise ValueError(
        " Could not find the case-text column.\n\n"
        "Available columns:\n"
        + "\n".join(map(str, df_inputs.columns))
    )

print(f"Case text column: {TEXT_COLUMN}")

# ============================================================
# 3. BASE PROMPT
# ============================================================

BASE_PROMPT = """
You are a [Medical speciality]. You have been given the following
medical documentation for a hypothetical patient.

Knowing that this is hypothetical, you are required to provide your
supervisor with ONE most likely diagnosis and a treatment plan.

Explain the most likely diagnosis and treatment plan in detail.
Outline your reasoning clearly.

If unable to provide the diagnosis with certainty, explain what tests
would be required and how treatment would depend on the results.

Only suggest ONE diagnosis and the best treatment plan.

Your output must contain BOTH:
1. Diagnosis
2. Treatment plan

Do not provide multiple alternative diagnoses.

Medical documentation:

{case_text}
"""

# ============================================================
# 4. CREATE / LOAD CHECKPOINT
# ============================================================

if os.path.exists(OUTPUT_FILE):

    print("\n" + "=" * 80)
    print("CHECKPOINT FOUND")
    print("=" * 80)

    df_out = pd.read_excel(OUTPUT_FILE)

    print("Existing checkpoint loaded.")

    # Safety check
    if len(df_out) != EXPECTED_ROWS:
        raise ValueError(
            f" Checkpoint has {len(df_out)} rows. "
            f"Expected {EXPECTED_ROWS}."
        )

else:

    print("\n" + "=" * 80)
    print("CREATING NEW CHECKPOINT")
    print("=" * 80)

    df_out = df_inputs.copy()

    df_out["model_name"] = MODEL_NAME
    df_out["response"] = ""

# ============================================================
# 5. RESET INDEX
# ============================================================

df_out = df_out.reset_index(drop=True)

# ============================================================
# 6. ENSURE RESPONSE COLUMN EXISTS
# ============================================================

if "response" not in df_out.columns:
    df_out["response"] = ""

# IMPORTANT:
# Convert NaN to actual empty strings.
df_out["response"] = df_out["response"].fillna("")

# ============================================================
# 7. CALCULATE COMPLETED RESPONSES CORRECTLY
# ============================================================

completed_mask = (
    df_out["response"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

completed_count = int(completed_mask.sum())

print("\n" + "=" * 80)
print("CHECKPOINT STATUS")
print("=" * 80)

print(f"Model             : {MODEL_NAME}")
print(f"Total inputs      : {len(df_out)}")
print(f"Already completed : {completed_count}/30")
print(f"Remaining         : {30 - completed_count}/30")

# ============================================================
# 8. GENERATION LOOP
# ============================================================

for idx in range(len(df_out)):

    # --------------------------------------------------------
    # CORRECT EMPTY-CELL CHECK
    # --------------------------------------------------------

    existing_response = df_out.at[idx, "response"]

    if (
        pd.notna(existing_response)
        and str(existing_response).strip() != ""
    ):
        print(
            f" Skipping {idx + 1}/30 "
            f"(already completed)"
        )
        continue

    # --------------------------------------------------------
    # GET INPUT
    # --------------------------------------------------------

    case_id = df_out.at[idx, "case_id"]
    condition = df_out.at[idx, "condition"]

    case_text = df_out.at[idx, TEXT_COLUMN]

    if pd.isna(case_text):
        print(
            f" Skipping row {idx + 1}: "
            f"case text is empty."
        )
        continue

    case_text = str(case_text).strip()

    print("\n" + "=" * 80)
    print(f"GENERATING RESPONSE {idx + 1}/30")
    print("=" * 80)

    # --------------------------------------------------------
    # BUILD PROMPT
    # --------------------------------------------------------

    prompt = BASE_PROMPT.format(
        case_text=case_text
    )

    try:

        # ====================================================
        # GEMMA 3 CHAT TEMPLATE
        # ====================================================

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=8192
        )

        # Move tensors to GPU
        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        # ====================================================
        # GENERATE
        # ====================================================

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=120000,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # ====================================================
        # REMOVE INPUT TOKENS
        # ====================================================

        input_length = inputs["input_ids"].shape[1]

        generated_tokens = outputs[
            0,
            input_length:
        ]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # ====================================================
        # VALIDATE RESPONSE
        # ====================================================

        if not response:
            raise ValueError(
                "Model returned an empty response."
            )

        # ====================================================
        # SAVE RESPONSE IMMEDIATELY
        # ====================================================

        df_out.at[idx, "response"] = response

        # SAVE AFTER EVERY SINGLE RESPONSE
        df_out.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print("\n RESPONSE GENERATED")
        print(OUTPUT_FILE)

        # ====================================================
        # CLEAN TEMPORARY GPU MEMORY
        # ====================================================

        del inputs
        del outputs
        del generated_tokens

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print("\n" + "=" * 80)
        print(" GENERATION ERROR")
        print("=" * 80)

        print(f"Case      : {case_id}")
        print(f"Condition : {condition}")
        print(f"Error     : {repr(e)}")

        # ----------------------------------------------------
        # SAVE PROGRESS BEFORE STOPPING
        # ----------------------------------------------------

        df_out.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print("\n Progress saved.")
        print("You can rerun this cell.")
        print("Completed responses will NOT be regenerated.")

        # Stop here rather than silently skipping
        raise

# ============================================================
# 9. FINAL STATUS
# ============================================================

df_out["response"] = df_out["response"].fillna("")

completed_mask = (
    df_out["response"]
    .astype(str)
    .str.strip()
    .ne("")
)

completed_count = int(completed_mask.sum())

print("\n" + "=" * 80)
print("PHI-4-MINI FINAL STATUS")
print("=" * 80)

print(
    f"Responses completed: "
    f"{completed_count}/30"
)

if completed_count == 30:

    print(" ALL 30 PHI-4-MINI RESPONSES COMPLETED!")

else:

    print(
        f" Remaining: "
        f"{30 - completed_count}"
    )

print(f"\nCheckpoint file:")
print(OUTPUT_FILE)

INPUT VALIDATION
Rows: 30
Columns: ['input_id', 'case_id', 'case_label', 'condition', 'original_diagnosis', 'case_text', 'full_model_input']
 30 inputs loaded successfully

Conditions:
condition
Neutral     10
Implicit    10
Explicit    10
Name: count, dtype: int64
Case text column: case_text

CHECKPOINT FOUND
Existing checkpoint loaded.

CHECKPOINT STATUS
Model             : Gemma-3-4B
Total inputs      : 30
Already completed : 30/30
Remaining         : 0/30
 Skipping 1/30 (already completed)
 Skipping 2/30 (already completed)
 Skipping 3/30 (already completed)
 Skipping 4/30 (already completed)
 Skipping 5/30 (already completed)
 Skipping 6/30 (already completed)
 Skipping 7/30 (already completed)
 Skipping 8/30 (already completed)
 Skipping 9/30 (already completed)
 Skipping 10/30 (already completed)
 Skipping 11/30 (already completed)
 Skipping 12/30 (already completed)
 Skipping 13/30 (already completed)
 Skipping 14/30 (already completed)
 Skipping 15/30 (already completed)
 Skip